# 01 · Inlist files: reading and writing

Everything in the setup starts with reading MESA inlists into Python dictionaries. That is `posydon/grids/setup/inlist.py`, and it does exactly one thing: text ⇄ `{section: {key: value}}`.

Two rules matter for everything that follows:

1. **Values stay strings, exactly as written.** `10d0` is not turned into a float, so writing it back gives MESA the same text.
2. **Keys keep the order they first appear in**, and a key written twice keeps its last value.

In [1]:
from nb_env import env, show
ENV = env()

from posydon.grids.setup.inlist import (
    LEGACY_SKIPPED_VALUES, format_value, parse_inlist, read_inlist,
    render_inlist, render_inlist_section, strip_comment)

## Parsing text

In [2]:
text = """
&controls
    initial_mass = 10d0      ! solar masses
    x_ctrl(1) = 0.5d0
    log_directory = 'LOGS!1' ! a "!" inside quotes is not a comment
    initial_mass = 12d0      ! written twice: the last one wins
/ ! end of controls

&star_job
    pgstar_flag = .false.
/
"""
parse_inlist(text)

{'controls': {'initial_mass': '12d0',
  'x_ctrl(1)': '0.5d0',
  'log_directory': "'LOGS!1'"},
 'star_job': {'pgstar_flag': '.false.'}}

`strip_comment` is the piece that handles `!` comments, and knows about quotes:

In [3]:
for line in ["initial_mass = 10d0  ! in Msun", "log_directory = 'LOGS!1'", "! only a comment"]:
    print(repr(line), "->", repr(strip_comment(line)))

'initial_mass = 10d0  ! in Msun' -> 'initial_mass = 10d0'
"log_directory = 'LOGS!1'" -> "log_directory = 'LOGS!1'"
'! only a comment' -> ''


## MESA's `.defaults` files have no sections

MESA's `controls.defaults`, `star_job.defaults`, … are plain `key = value` lines with no `&controls` line. Pass `default_section` to say which section they belong to.

In [4]:
defaults = ENV.inlists / "r11701" / "mesa" / "default" / "defaults"
controls = read_inlist(defaults / "star" / "controls.defaults", default_section="controls")
print(len(controls["controls"]), "keys in MESA's controls.defaults")
list(controls["controls"].items())[:8]

1678 keys in MESA's controls.defaults


[('initial_mass', '1'),
 ('initial_z', '0.02d0'),
 ('initial_y', '-1'),
 ('initial_he3', '-1'),
 ('terminal_interval', '1'),
 ('write_header_frequency', '10'),
 ('extra_terminal_output_file', "''"),
 ('terminal_show_age_in_years', '.true.')]

Two odd things live in MESA's defaults, and the setup keeps today's handling of both:

- some values are the empty string `''` or `'.'`: today's setup drops them (`LEGACY_SKIPPED_VALUES`), so MESA uses its compiled-in default;
- array keys are written as `x_ctrl(1:num_x_ctrls)`, which MESA cannot read back. Notebook 02 shows how that is fixed (`fix_num_x_ctrls`, tracker C2).

In [5]:
print("skipped values:", LEGACY_SKIPPED_VALUES)
all_keys = controls["controls"]
skipped = {k: v for k, v in all_keys.items() if v in LEGACY_SKIPPED_VALUES}
print(len(skipped), "keys would be skipped, e.g.", list(skipped.items())[:3])
print([k for k in all_keys if "num_x_ctrls" in k])

skipped values: ("''", "'.'")
21 keys would be skipped, e.g. [('extra_terminal_output_file', "''"), ('trace_history_value_name(:)', "''"), ('star_history_header_name', "''")]
['x_ctrl(1:num_x_ctrls)', 'x_integer_ctrl(1:num_x_ctrls)', 'x_logical_ctrl(1:num_x_ctrls)']


In [6]:
kept = read_inlist(defaults / "star" / "controls.defaults", default_section="controls",
                   skip_values=LEGACY_SKIPPED_VALUES)
print(len(all_keys), "->", len(kept["controls"]), "keys after skipping")

1678 -> 1657 keys after skipping


## A real POSYDON inlist

`common/binary/inlist1` of release v2.0 is the star-1 inlist every v2 grid starts from. It has both sections.

In [7]:
inlist1 = read_inlist(ENV.release / "common" / "binary" / "inlist1")
{section: len(values) for section, values in inlist1.items()}

{'star_job': 27, 'controls': 131}

## Writing

`format_value` turns Python values into Fortran text; strings go through untouched (so they must carry their own quotes). `render_inlist` writes sections back, and `parse_inlist` reads that back to the same dictionary.

In [8]:
for value in [True, False, 3, 3.0, 12.345, "'LOGS1'", "1d-3"]:
    print(f"{value!r:10} -> {format_value(value)}")

True       -> .true.
False      -> .false.
3          -> 3
3.0        -> 3
12.345     -> 1.2345000000d+01
"'LOGS1'"  -> 'LOGS1'
'1d-3'     -> 1d-3


In [9]:
print(render_inlist_section("star_job", {"pgstar_flag": False, "save_model_filename": "'final.mod'"}))

&star_job
    pgstar_flag = .false.
    save_model_filename = 'final.mod'
/



In [10]:
sections = parse_inlist(text)
assert parse_inlist(render_inlist(sections)) == sections
print(render_inlist(sections))

&controls
    initial_mass = 12d0
    x_ctrl(1) = 0.5d0
    log_directory = 'LOGS!1'
/

&star_job
    pgstar_flag = .false.
/



> **Note.** The run directory's inlists are written by `build.py` in today's exact layout (tabs, `/ ! end of ... namelist` lines) so the goldens stay byte-identical; `render_inlist` is the clean writer for everything else.

## Try it

- Read `ENV.release / "CO-HeMS" / "star1_formation" / "inlist_step1"` and look at what the first formation step changes.
- Count how many keys of `common/binary/inlist1`'s `&controls` differ from MESA's defaults:

In [11]:
mesa = kept["controls"]
posydon = inlist1["controls"]
changed = {k: (mesa.get(k), v) for k, v in posydon.items() if mesa.get(k) != v}
print(len(changed), "of", len(posydon), "keys in inlist1 differ from MESA's defaults; a few:")
for key, (old, new) in list(changed.items())[:10]:
    print(f"  {key:40} {old!s:>12} -> {new}")

114 of 131 keys in inlist1 differ from MESA's defaults; a few:
  warn_when_large_rel_run_E_err                  0.02d0 -> 1d99
  warn_when_stop_checking_residuals              .true. -> .false.
  use_gold_tolerances                            .true. -> .false.
  use_eosELM                                     .true. -> .false.
  use_eosDT2                                     .true. -> .false.
  photo_directory                              'photos' -> 'photos1'
  log_directory                                  'LOGS' -> 'LOGS1'
  history_interval                                    5 -> 1
  terminal_interval                                   1 -> 10
  write_header_frequency                             10 -> 100
